# Ticket Priority Prediction

## Objective

Build a machine learning model to predict ticket priority (Low, Medium, High) from customer support ticket information.

In [1]:
import pandas as pd
import joblib

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

In [2]:
df = pd.read_csv("../data/raw/customer_support_tickets.csv")

print(df.shape)
df.head()

(8469, 17)


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [3]:
selected_columns = [
    "Ticket Subject",
    "Ticket Description",
    "Ticket Type",
    "Product Purchased",
    "Ticket Channel",
    "Customer Age",
    "Customer Gender",
    "Ticket Priority",
]

df = df[selected_columns].copy()

print(df.shape)
df.head()

(8469, 8)


,Ticket Subject,Ticket Description,Ticket Type,Product Purchased,Ticket Channel,Customer Age,Customer Gender,Ticket Priority
0,Product setup,I'm having an issue with the {product_purchase...,Technical issue,GoPro Hero,Social media,32,Other,Critical
1,Peripheral compatibility,I'm having an issue with the {product_purchase...,Technical issue,LG Smart TV,Chat,42,Female,Critical
2,Network problem,I'm facing a problem with my {product_purchase...,Technical issue,Dell XPS,Social media,48,Other,Low
3,Account access,I'm having an issue with the {product_purchase...,Billing inquiry,Microsoft Office,Social media,27,Female,Low
4,Data loss,I'm having an issue with the {product_purchase...,Billing inquiry,Autodesk AutoCAD,Email,67,Female,Low


In [4]:
print(df.isnull().sum())

Ticket Subject        0
Ticket Description    0
Ticket Type           0
Product Purchased     0
Ticket Channel        0
Customer Age          0
Customer Gender       0
Ticket Priority       0
dtype: int64


# Create Combined Text Feature

In [5]:
df["Ticket Text"] = (
    df["Ticket Subject"].astype(str)
    + " "
    + df["Ticket Description"].astype(str)
)

df[["Ticket Subject", "Ticket Description", "Ticket Text"]].head()

,Ticket Subject,Ticket Description,Ticket Text
0,Product setup,I'm having an issue with the {product_purchase...,Product setup I'm having an issue with the {pr...
1,Peripheral compatibility,I'm having an issue with the {product_purchase...,Peripheral compatibility I'm having an issue w...
2,Network problem,I'm facing a problem with my {product_purchase...,Network problem I'm facing a problem with my {...
3,Account access,I'm having an issue with the {product_purchase...,Account access I'm having an issue with the {p...
4,Data loss,I'm having an issue with the {product_purchase...,Data loss I'm having an issue with the {produc...


# Encode Categorical Features

In [6]:
label_encoders = {}

categorical_columns = [
    "Ticket Type",
    "Product Purchased",
    "Ticket Channel",
    "Customer Gender",
    "Ticket Priority",
]

for column in categorical_columns:
    encoder = LabelEncoder()
    df[column] = encoder.fit_transform(df[column])
    label_encoders[column] = encoder

print("Categorical features encoded successfully.")

Categorical features encoded successfully.


In [7]:
print("Unique encoded Ticket Priority values:")
print(sorted(df["Ticket Priority"].unique()))

print("\nClass distribution:")
print(df["Ticket Priority"].value_counts())

Unique encoded Ticket Priority values:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3)]

Class distribution:
Ticket Priority
3    2192
0    2129
1    2085
2    2063
Name: count, dtype: int64


# TF-IDF Feature Extraction

In [8]:
tfidf = TfidfVectorizer(
    max_features=1000,
    stop_words="english"
)

X_text = tfidf.fit_transform(df["Ticket Text"])

print("TF-IDF Matrix Shape:", X_text.shape)

TF-IDF Matrix Shape: (8469, 1000)


In [9]:
numeric_features = df[
    [
        "Ticket Type",
        "Product Purchased",
        "Ticket Channel",
        "Customer Age",
        "Customer Gender",
    ]
]

print(numeric_features.shape)
numeric_features.head()

(8469, 5)


,Ticket Type,Product Purchased,Ticket Channel,Customer Age,Customer Gender
0,4,16,3,32,2
1,4,21,0,42,0
2,4,10,3,48,2
3,0,25,3,27,0
4,0,5,1,67,0


In [10]:
X = hstack([X_text, numeric_features.values])

y = df["Ticket Priority"]

print("Final Feature Matrix:", X.shape)
print("Target Shape:", y.shape)

Final Feature Matrix: (8469, 1005)
Target Shape: (8469,)


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (6775, 1005)
X_test : (1694, 1005)
y_train: (6775,)
y_test : (1694,)


# Import Classification Models

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# Logistic Regression

In [13]:
logistic_model = LogisticRegression(
    max_iter=3000,
    random_state=42
)

logistic_model.fit(X_train, y_train)

logistic_pred = logistic_model.predict(X_test)

logistic_accuracy = accuracy_score(y_test, logistic_pred)

print(f"Accuracy: {logistic_accuracy:.4f}")

print("\nClassification Report")
print(classification_report(y_test, logistic_pred))

Accuracy: 0.2627

Classification Report
              precision    recall  f1-score   support

           0       0.26      0.28      0.27       426
           1       0.28      0.27      0.27       417
           2       0.24      0.23      0.23       413
           3       0.27      0.28      0.28       438

    accuracy                           0.26      1694
   macro avg       0.26      0.26      0.26      1694
weighted avg       0.26      0.26      0.26      1694



C:\Users\User\Desktop\QResolve\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:470: ConvergenceWarning: lbfgs failed to converge after 3000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=3000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


# Decision Tree Classifier

In [14]:
dt_model = DecisionTreeClassifier(random_state=42)

dt_model.fit(X_train, y_train)

dt_pred = dt_model.predict(X_test)

dt_accuracy = accuracy_score(y_test, dt_pred)

print(f"Accuracy: {dt_accuracy:.4f}")

print("\nClassification Report")
print(classification_report(y_test, dt_pred))

Accuracy: 0.2586

Classification Report
              precision    recall  f1-score   support

           0       0.25      0.23      0.24       426
           1       0.26      0.27      0.27       417
           2       0.26      0.27      0.27       413
           3       0.25      0.26      0.26       438

    accuracy                           0.26      1694
   macro avg       0.26      0.26      0.26      1694
weighted avg       0.26      0.26      0.26      1694



In [15]:
priority_encoder = label_encoders["Ticket Priority"]

print(priority_encoder.classes_)

['Critical' 'High' 'Low' 'Medium']


# Random Forest Classifier

In [16]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)

print(f"Accuracy: {rf_accuracy:.4f}")

print("\nClassification Report")
print(classification_report(y_test, rf_pred))

Accuracy: 0.2527

Classification Report
              precision    recall  f1-score   support

           0       0.27      0.26      0.26       426
           1       0.26      0.26      0.26       417
           2       0.23      0.20      0.21       413
           3       0.25      0.29      0.27       438

    accuracy                           0.25      1694
   macro avg       0.25      0.25      0.25      1694
weighted avg       0.25      0.25      0.25      1694



# Gradient Boosting Classifier

In [17]:
gb_model = GradientBoostingClassifier(random_state=42)

gb_model.fit(X_train, y_train)

gb_pred = gb_model.predict(X_test)

gb_accuracy = accuracy_score(y_test, gb_pred)

print(f"Accuracy: {gb_accuracy:.4f}")

print("\nClassification Report")
print(classification_report(y_test, gb_pred))

Accuracy: 0.2627

Classification Report
              precision    recall  f1-score   support

           0       0.24      0.23      0.23       426
           1       0.28      0.23      0.25       417
           2       0.25      0.14      0.18       413
           3       0.27      0.45      0.34       438

    accuracy                           0.26      1694
   macro avg       0.26      0.26      0.25      1694
weighted avg       0.26      0.26      0.25      1694



In [18]:
for priority in sorted(df["Ticket Priority"].unique()):
    print("=" * 60)
    print("Priority:", label_encoders["Ticket Priority"].inverse_transform([priority])[0])
    print()

    samples = df[df["Ticket Priority"] == priority]["Ticket Text"].head(5)

    for i, text in enumerate(samples, 1):
        print(f"{i}. {text}")
        print()

Priority: Critical

1. Product setup I'm having an issue with the {product_purchased}. Please assist.

Your billing zip code is: 71701.

We appreciate that you have requested a website address.

Please double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists.

2. Peripheral compatibility I'm having an issue with the {product_purchased}. Please assist.

If you need to change an existing product.

I'm having an issue with the {product_purchased}. Please assist.

If The issue I'm facing is intermittent. Sometimes it works fine, but other times it acts up unexpectedly.

3. Refund request I'm unable to access my {product_purchased} account. It keeps displaying an 'Invalid Credentials' error, even though I'm using the correct login information. How can I regain access to my account?

Solution 1 I'm unable to find the option to perform the desired action in the {product_purchased}. Could you please guide me through the steps?

4. B